In [2]:
%%writefile model.py
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=100, features=64):
        super().__init__()
        self.net = nn.Sequential(
            self._block(z_dim, features * 4, 4, 1, 0),
            self._block(features * 4, features * 2, 4, 2, 1),
            self._block(features * 2, features, 4, 2, 1),
            nn.ConvTranspose2d(features, 3, 4, 2, 1),
            nn.Tanh()
        )

    def _block(self, in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(True)
        )

    def forward(self, x):
        return self.net(x)


class Critic(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            self._block(features, features * 2, 4, 2, 1),
            self._block(features * 2, features * 4, 4, 2, 1),
            nn.Conv2d(features * 4, 1, 4, 1, 0)
        )

    def _block(self, in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        return self.net(x).view(-1)

Writing model.py


In [3]:
import os
import json
import torch
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from model import Generator, Critic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 5e-5
batch_size = 64
image_size = 32
channels_img = 3
z_dim = 100
num_epochs = 50
features_d = 64
features_g = 64
critic_iterations = 5
weight_clip = 0.01

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5 for _ in range(channels_img)], [0.5 for _ in range(channels_img)])
])

dataset = torchvision.datasets.CIFAR10(root="dataset/", train=True, transform=transform, download=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

gen = Generator(z_dim, features_g).to(device)
critic = Critic(features_d).to(device)

opt_gen = optim.RMSprop(gen.parameters(), lr=lr)
opt_critic = optim.RMSprop(critic.parameters(), lr=lr)

loss_logs = {"critic_loss": [], "gen_loss": []}

for epoch in range(num_epochs):
    for batch_idx, (data, _) in enumerate(loader):
        data = data.to(device)
        cur_batch_size = data.shape[0]

        for _ in range(critic_iterations):
            noise = torch.randn(cur_batch_size, z_dim, 1, 1).to(device)
            fake = gen(noise)
            critic_real = critic(data)
            critic_fake = critic(fake.detach())

            loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))

            opt_critic.zero_grad()
            loss_critic.backward()
            opt_critic.step()

            for p in critic.parameters():
                p.data.clamp_(-weight_clip, weight_clip)

        fake = gen(noise)
        output = critic(fake)
        loss_gen = -torch.mean(output)

        opt_gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss C: {loss_critic:.4f} | Loss G: {loss_gen:.4f}")
    loss_logs["critic_loss"].append(loss_critic.item())
    loss_logs["gen_loss"].append(loss_gen.item())

torch.save(gen.state_dict(), "generator.pth")
torch.save(critic.state_dict(), "critic.pth")

with open("loss_log.json", "w") as f:
    json.dump(loss_logs, f)

100%|██████████| 170M/170M [00:04<00:00, 41.4MB/s]


Epoch [1/50] | Loss C: -0.5527 | Loss G: 0.2805
Epoch [2/50] | Loss C: -0.4800 | Loss G: 0.2483
Epoch [3/50] | Loss C: -0.4862 | Loss G: 0.2474
Epoch [4/50] | Loss C: -0.3902 | Loss G: 0.2466
Epoch [5/50] | Loss C: -0.3918 | Loss G: 0.1981
Epoch [6/50] | Loss C: -0.3694 | Loss G: 0.2166
Epoch [7/50] | Loss C: -0.3309 | Loss G: 0.1246
Epoch [8/50] | Loss C: -0.3016 | Loss G: 0.1645
Epoch [9/50] | Loss C: -0.2881 | Loss G: 0.1378
Epoch [10/50] | Loss C: -0.3410 | Loss G: 0.1421
Epoch [11/50] | Loss C: -0.3437 | Loss G: 0.1994
Epoch [12/50] | Loss C: -0.2166 | Loss G: 0.1372
Epoch [13/50] | Loss C: -0.2936 | Loss G: 0.1871
Epoch [14/50] | Loss C: -0.2542 | Loss G: 0.1188
Epoch [15/50] | Loss C: -0.3143 | Loss G: 0.1309
Epoch [16/50] | Loss C: -0.3084 | Loss G: 0.2007
Epoch [17/50] | Loss C: -0.2380 | Loss G: 0.1055
Epoch [18/50] | Loss C: -0.2506 | Loss G: 0.1458
Epoch [19/50] | Loss C: -0.1830 | Loss G: 0.2117
Epoch [20/50] | Loss C: -0.2004 | Loss G: 0.1542
Epoch [21/50] | Loss C: -0.17